# Noise sweep against SATIRE-S (sweep_noise.dag)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_N = '/data/spotless/scratch/sstucki/starsim/noise'
TAG = 'noise_%s_m45s_size_4096_N60'
SATIRE = pd.read_csv('/nfs/pic.es/user/s/sstucki/sunsim/PIC/satire_ff.csv', parse_dates=['date'])
X = {'plage': 'A_th', 'fraction': 'B_sat'}
KEYS = ['noise_th', 'B_th', 'I_th']
I_SHOW = 0.9                                     # I_th of the figures of the faculae
SIGMA_COLOURS = {8: '#2a78d6', 20: '#eb6834', 50: '#1baf7a', 100: '#eda100'}
SIGMA_MARKERS = {8: 'o', 20: 's', 50: '^', 100: 'D'}
PERIODS = {'2017': ('2017-06-28', '2017-09-03'), '2021': ('2021-03-29', '2021-08-03')}


def load(kind):
    """satire_ff.csv of collect_noise.sh with the RV / TSI statistics of sweep_stats.csv (one row per set)."""
    res = os.path.join(BASE_N, 'results_' + TAG % kind)
    ff = pd.read_csv(os.path.join(res, 'satire_ff.csv'))
    st = pd.read_csv(os.path.join(res, 'sweep_stats.csv'))
    keys = KEYS + [X[kind]]
    for d in (ff, st):
        d[keys] = d[keys].astype(float).round(4)
    return ff.merge(st, on=keys, how='left')


df = {kind: load(kind) for kind in X}
for kind, d in df.items():
    print(kind, len(d), 'sets,', d['n_fc'].max(), 'SATIRE days')

## Tables

In [ ]:
COLS = ['noise_th', 'B_th', 'I_th', 'ratio_sp', 'rms_sp', 'corr_sp', 'ratio_fc', 'rms_fc', 'corr_fc',
        'rv_rms', 'sorce_rms', 'tsis_rms']
for kind, d in df.items():
    cols = COLS[:3] + [X[kind]] + COLS[3:] + (['ratio_fc_same', 'rms_fc_same'] if kind == 'fraction' else [])
    print(kind)
    display(d.sort_values('rms_fc')[[c for c in cols if c in d]].round(4))

## Faculae, fraction

In [ ]:
d = df['fraction'][np.isclose(df['fraction']['I_th'], I_SHOW)]
panels = [('ratio_fc', 'ratio of the means, SATIRE 305 G'), ('rms_fc', 'rms of the difference [%], SATIRE 305 G'),
          ('ratio_fc_same', 'ratio of the means, SATIRE same B_sat')]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, (value, label) in zip(axes, panels):
    for sigma, g in d.groupby('noise_th'):
        g = g.sort_values('B_sat')
        ax.plot(g['B_sat'], g[value], marker=SIGMA_MARKERS[sigma], ms=8, lw=2, c=SIGMA_COLOURS[sigma],
                label='σ = %g G (B_th = %g G)' % (sigma, 3 * sigma))
    if value.startswith('ratio'):
        ax.axhline(1, c='0.5', lw=1, ls='--')
    ax.set_xscale('log')
    ax.set_xticks([25, 50, 100, 200, 300, 500], ['25', '50', '100', '200', '300', '500'])
    ax.set_xlabel('B_sat [G]')
    ax.set_ylabel(label)
axes[0].legend()
fig.suptitle('Facular filling factor against SATIRE-S (I_th = %g)' % I_SHOW)
fig.tight_layout()
plt.show()

## Faculae, plage A_th = 0

In [ ]:
d = df['plage']
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, value, label in zip(axes, ('ratio_fc', 'rms_fc', 'corr_fc'),
                            ('ratio of the means', 'rms of the difference [%]', 'correlation')):
    for sigma, g in d.groupby('noise_th'):
        g = g.sort_values('I_th')
        ax.plot(g['I_th'], g[value], marker=SIGMA_MARKERS[sigma], ms=8, lw=2, c=SIGMA_COLOURS[sigma],
                label='σ = %g G (B_th = %g G)' % (sigma, 3 * sigma))
    if value == 'ratio_fc':
        ax.axhline(1, c='0.5', lw=1, ls='--')
    ax.set_xlabel('I_th')
    ax.set_ylabel(label)
axes[0].legend()
fig.suptitle('Facular filling factor against SATIRE-S 305 G (plage, A_th = 0)')
fig.tight_layout()
plt.show()

## Spots

In [ ]:
d = df['plage']                                  # the spots do not depend on the kind of faculae
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, value, label in zip(axes, ('ratio_sp', 'rms_sp', 'corr_sp'),
                            ('ratio of the means', 'rms of the difference [%]', 'correlation')):
    for sigma, g in d.groupby('noise_th'):
        g = g.sort_values('I_th')
        ax.plot(g['I_th'], g[value], marker=SIGMA_MARKERS[sigma], ms=8, lw=2, c=SIGMA_COLOURS[sigma],
                label='σ = %g G (B_th = %g G)' % (sigma, 3 * sigma))
    if value == 'ratio_sp':
        ax.axhline(1, c='0.5', lw=1, ls='--')
    ax.set_xlabel('I_th')
    ax.set_ylabel(label)
axes[0].legend()
fig.suptitle('Spot filling factor against SATIRE-S (umbra + penumbra)')
fig.tight_layout()
plt.show()

## RV and TSI against the agreement with SATIRE

In [ ]:
for kind, d in df.items():
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharex=True)
    for ax, value, label in zip(axes, ('rv_rms', 'sorce_rms', 'tsis_rms'),
                                ('RV rms vs HARPS-N [m/s]', 'TSI rms vs SORCE [ppm]', 'TSI rms vs TSIS-1 [ppm]')):
        if value not in d:
            continue
        for sigma, g in d.groupby('noise_th'):
            ax.plot(g['ratio_fc'], g[value], SIGMA_MARKERS[sigma], ms=8, c=SIGMA_COLOURS[sigma], alpha=0.8,
                    mec='white', mew=1, label='σ = %g G' % sigma)
        ax.axvline(1, c='0.5', lw=1, ls='--')
        ax.set_xlabel('ff_fc / ff_fc SATIRE 305 G')
        ax.set_ylabel(label)
    axes[0].legend()
    fig.suptitle(kind)
    fig.tight_layout()
    plt.show()

## Time series on the SATIRE days

In [ ]:
def summary(kind, label):
    s = pd.read_csv(os.path.join(BASE_N, 'sets_' + TAG % kind, 'ident', label, 'summary.csv'))
    s['date'] = pd.to_datetime(s['date'].astype(str).str[:10])
    return s


# (kind, set label): by default the best facular match and the best RV of every kind
CHOSEN = []
for kind, d in df.items():
    for value in ('rms_fc', 'rv_rms'):
        if value in d and d[value].notna().any():
            CHOSEN.append((kind, d.sort_values(value).iloc[0]['set']))
CHOSEN = list(dict.fromkeys(CHOSEN))

for kind, label in CHOSEN:
    s = summary(kind, label)
    row = df[kind].set_index('set').loc[label]
    fig, axes = plt.subplots(2, len(PERIODS), figsize=(16, 6), sharex='col', sharey='row')
    for c, (period, (t0, t1)) in enumerate(PERIODS.items()):
        ss = s[(s['date'] >= t0) & (s['date'] <= t1)]
        sat = SATIRE[(SATIRE['date'] >= t0) & (SATIRE['date'] <= t1)]
        axes[0, c].plot(ss['date'], ss['ff_sp'], 'o-', ms=4, lw=1.5, c='#2a78d6', label='SDO')
        axes[0, c].plot(sat['date'], sat['ff_sp'], 's-', ms=4, lw=1.5, c='#eb6834', label='SATIRE')
        axes[1, c].plot(ss['date'], ss['ff_fc'], 'o-', ms=4, lw=1.5, c='#2a78d6', label='SDO')
        axes[1, c].plot(sat['date'], sat['ff_fc_305'], 's-', ms=4, lw=1.5, c='#eb6834', label='SATIRE 305 G')
        if kind == 'fraction' and 'ff_fc_%g' % row['B_sat'] in sat and row['B_sat'] != 305:
            axes[1, c].plot(sat['date'], sat['ff_fc_%g' % row['B_sat']], '^--', ms=4, lw=1.5, c='#1baf7a',
                            label='SATIRE %g G' % row['B_sat'])
        axes[0, c].set_title(period)
        axes[1, c].tick_params(axis='x', rotation=30)
    axes[0, 0].set_ylabel('ff spots [%]')
    axes[1, 0].set_ylabel('ff faculae [%]')
    axes[0, 0].legend()
    axes[1, 0].legend()
    fig.suptitle('%s: %s (rms_fc = %.3f %%, ratio_fc = %.2f)' % (kind, label, row['rms_fc'], row['ratio_fc']))
    fig.tight_layout()
    plt.show()